In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from PIL import Image
from torchvision.transforms import v2
from torchinfo import summary
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize
from imblearn.metrics import sensitivity_score, specificity_score
import pandas as pd
import numpy as np
import os

**Enable cuda if available**

In [32]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [33]:
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [34]:
writer = SummaryWriter()

In [35]:
class ISIC2019(Dataset): # TODO: consider maybe removing downsampled or removing duplicates. unsure if these are necessary
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_labels.drop("UNK", axis=1, inplace=True) # remove unknown category
        self.ohe_labels = self.img_labels.iloc[:, 1:]

        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f"{self.img_labels.iloc[idx, 0]}.jpg")
        image = Image.open(img_path)
        label = np.where(self.ohe_labels==1)[1][idx]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

In [36]:
transform = v2.Compose([
    v2.Resize((224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])

In [37]:
isic_dataset = ISIC2019(img_dir='../data/ISIC_2019_Training_Input', annotations_file='../data/ISIC_2019_Training_GroundTruth.csv', transform=transform)
train_images_size = len(isic_dataset)

In [38]:
train_size = int(train_images_size * 0.80)
test_size = train_images_size - train_size
# 80% train 20% test

isic_train, isic_test = random_split(isic_dataset, [train_size, test_size])
train_size, test_size

(20264, 5067)

In [39]:
class MobileNetV3_Baseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_classes = 8
        self.mobilenet = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT) # use imagenet pretrained weights
        in_features = self.mobilenet.classifier[3].in_features
        self.mobilenet.classifier[3] = nn.Linear(in_features, self.num_classes) # change number of output categories

        nn.init.normal_(self.mobilenet.classifier[3].weight, 0, 0.01) 
        nn.init.zeros_(self.mobilenet.classifier[3].bias) # use same weight + bias init as pytorch for consistency

    def forward(self, x):
        x = self.mobilenet(x)
        return x

In [40]:
model = MobileNetV3_Baseline()

**Move model to specified device**

In [41]:
model = model.to(device=device)

In [ ]:
epochs = 100
learning_rate = 1e-4
batch_size = 32

start_epoch = 0 # epoch number of checkpoint to load

if start_epoch > 0:
    model.load_state_dict(torch.load(f"checkpoints/baseline/epoch-{start_epoch}.pth")) # load specific checkpoint

output_dir = "checkpoints/baseline"
os.makedirs(output_dir, exist_ok=True) # create baseline dir for checkpoints if not exist

# TODO - find better parameters for baseline

In [ ]:
summary(model, input_size=(batch_size, 3, 224, 224))

Layer (type:depth-idx)                                  Output Shape              Param #
MobileNetV3_Baseline                                    [32, 8]                   --
├─MobileNetV3: 1-1                                      [32, 8]                   --
│    └─Sequential: 2-1                                  [32, 960, 7, 7]           --
│    │    └─Conv2dNormActivation: 3-1                   [32, 16, 112, 112]        464
│    │    └─InvertedResidual: 3-2                       [32, 16, 112, 112]        464
│    │    └─InvertedResidual: 3-3                       [32, 24, 56, 56]          3,440
│    │    └─InvertedResidual: 3-4                       [32, 24, 56, 56]          4,440
│    │    └─InvertedResidual: 3-5                       [32, 40, 28, 28]          10,328
│    │    └─InvertedResidual: 3-6                       [32, 40, 28, 28]          20,992
│    │    └─InvertedResidual: 3-7                       [32, 40, 28, 28]          20,992
│    │    └─InvertedResidual: 3-8       

In [44]:
dataloader_train = DataLoader(isic_train, batch_size=batch_size, shuffle=True, pin_memory=True)
dataloader_test = DataLoader(isic_test, batch_size=batch_size, shuffle=False, pin_memory=True)

num_train_batches = len(dataloader_train)
num_test_batches = len(dataloader_test)

loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
num_train_batches, num_test_batches

(634, 159)

In [ ]:
for epoch in range(start_epoch, epochs):
    train_epoch_preds = []
    train_epoch_labels = []
    train_epoch_probs  = []
    test_epoch_preds = []
    test_epoch_labels = []
    test_epoch_probs  = []

    model.train()
    for batch_idx, (train_features, train_labels) in enumerate(dataloader_train):
        train_features = train_features.to(device)
        train_labels = train_labels.to(device) # move to device

        optimizer.zero_grad()

        predictions = model(train_features)
        probs = torch.softmax(predictions, dim=1) # compute probabilities
        predictions_labels = torch.argmax(predictions, dim=1)

        train_epoch_preds.extend(predictions_labels.cpu().numpy()) # append current predictions
        train_epoch_probs.extend(probs.detach().cpu().numpy()) # append current probabilities
        train_epoch_labels.extend(train_labels.cpu().numpy()) # append current labels

        train_batch_loss = loss(predictions, train_labels)
        train_batch_loss.backward()

        optimizer.step()

        train_loss += train_batch_loss.item()

    print(f"Saving epoch {epoch+1}...")
    torch.save(model.state_dict(), f"checkpoints/baseline/epoch-{epoch+1}.pth") # checkpoint per epoch for safety

    test_loss = 0

    model.eval()
    with torch.no_grad():
        for batch_idx, (test_features, test_labels) in enumerate(dataloader_test):
            test_features = test_features.to(device)
            test_labels = test_labels.to(device) # move to device
            
            predictions = model(test_features)
            probs = torch.softmax(predictions, dim=1) # compute probabilities
            predictions_labels = torch.argmax(predictions, dim=1)

            test_epoch_preds.extend(predictions_labels.cpu().numpy()) # append current predictions
            test_epoch_probs.extend(probs.detach().cpu().numpy()) # append current probabilities
            test_epoch_labels.extend(test_labels.cpu().numpy()) # append current labels

            test_batch_loss = loss(predictions, test_labels)
            test_loss += test_batch_loss.item()

    y_train_score = np.vstack(train_epoch_probs)
    y_train_onehot = label_binarize(train_epoch_labels, classes=np.arange(model.num_classes))

    y_test_score = np.vstack(test_epoch_probs)
    y_test_onehot = label_binarize(test_epoch_labels, classes=np.arange(model.num_classes))

    train_epoch_acc = accuracy_score(train_epoch_labels, train_epoch_preds)
    train_epoch_prec = precision_score(train_epoch_labels, train_epoch_preds, average='macro')
    train_epoch_rec = recall_score(train_epoch_labels, train_epoch_preds, average='macro')
    train_epoch_f1 = f1_score(train_epoch_labels, train_epoch_preds, average='macro')
    train_epoch_auc = roc_auc_score(y_train_onehot, y_train_score, average='macro', multi_class='ovr')
    train_epoch_spec = specificity_score(train_epoch_labels, train_epoch_preds, average='macro')

    train_loss /= num_train_batches
    test_loss /= num_test_batches

    test_epoch_acc = accuracy_score(test_epoch_labels, test_epoch_preds)
    test_epoch_prec = precision_score(test_epoch_labels, test_epoch_preds, average='macro')
    test_epoch_rec = recall_score(test_epoch_labels, test_epoch_preds, average='macro')
    test_epoch_f1 = f1_score(test_epoch_labels, test_epoch_preds, average='macro')
    test_epoch_auc = roc_auc_score(y_test_onehot, y_test_score, average='macro', multi_class='ovr')
    test_epoch_spec = specificity_score(test_epoch_labels, test_epoch_preds, average='macro')

    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_epoch_acc, epoch)
    writer.add_scalar('Precision/train', train_epoch_prec, epoch)
    writer.add_scalar('Recall-Sensitivity/train', train_epoch_rec, epoch)
    writer.add_scalar('AUC/train', train_epoch_auc, epoch)
    writer.add_scalar('Specificity/train', train_epoch_spec, epoch)

    writer.add_scalar("Loss/test", test_loss, epoch)
    writer.add_scalar('Accuracy/test', test_epoch_acc, epoch)
    writer.add_scalar('Precision/test', test_epoch_prec, epoch)
    writer.add_scalar('Recall-Sensitivity/test', test_epoch_rec, epoch)
    writer.add_scalar('AUC/test', test_epoch_auc, epoch)
    writer.add_scalar('Specificity/test', test_epoch_spec, epoch)

torch.Size([32]) torch.Size([32])
torch.Size([32]) torch.Size([32])
torch.Size([32]) torch.Size([32])


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), f"checkpoints/baseline/final_checkpoint.pth") # last checkpoint after model is done training

In [ ]:
writer.flush()